In [1]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from csi_vae_gumbel import util
from csi_vae_gumbel.dataset.load_datasets import load_datasets
from csi_vae_gumbel.models import vae
from csi_vae_gumbel.settings import Settings

settings = Settings()

/mnt/servicesdata/lcotti/csi-vae-gumbel/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_ds, test_ds = load_datasets(
    dataset_path=Path(f"../{settings.dataset_path}"),
    window_size=settings.test_window_size,  # Window size here is (usually) different from VAE training
    test_ratio=settings.test_ratio,
    n_activities=settings.n_activities,
    n_antennas=settings.n_antennas,
    antenna_select=settings.antenna_select,
    seed=settings.seed,
)
train_dl = DataLoader(
    train_ds,
    batch_size=settings.train_batch_size,
    shuffle=False,
    pin_memory=True,
)

In [4]:
best_model_path = util.get_best_model_path(Path(f"../{settings.study_path}"))
best_params = vae.Parameters(**util.get_vae_params(best_model_path))

vae_model = vae.MultiAntennaVAE(
    settings.n_antennas,
    settings.train_window_size,
    settings.n_subcarriers,
    settings.n_categories,
    best_params.latent_dim,
)
best_model_weights = torch.load(best_model_path / "model.pt", weights_only=True)
vae_model.load_state_dict(best_model_weights)

FileNotFoundError: [Errno 2] No such file or directory: '../out/a4_w75_tw450_b512/study_results.json'

In [ ]:
vae_model.tsne(train_dl)